# NumPy: thinking in arrays

A set of sensor readings needs calibration, filtering and daily summaries. Before choosing an array operation, write down what each axis represents. Predict the output shape, calculate one value by hand, and use that small check to catch operations that are syntactically valid but answer a different question.

Imagine measuring the temperature every hour for a month. You could store each reading in a Python list, but questions quickly become repetitive: add 2 degrees to every reading, keep only hot hours, calculate the daily average, or compare two sensors. Writing a loop for each operation is possible, but it gets noisy and easy to get wrong.

NumPy gives us arrays: compact collections of values that support whole-array numerical operations. Instead of describing how to process each value one at a time, we can describe the operation and let NumPy apply it across the data.

<div style="background-color:#eaf4fb; border-left:5px solid #2878a5; padding:10px 14px; margin:12px 0;">
<strong>The same calculation needs to apply to many measurements.</strong><br>
NumPy is useful when many values should follow the same numerical rule, while their positions and shape still carry meaning.
</div>

## 1. From a Python list to a numerical array

A Python list can store different kinds of objects. That flexibility is helpful for general programming, but numerical work usually needs a regular collection of values that behave consistently. A NumPy array is designed for that job.

Start with five temperatures: 18, 20, 21, 19, and 22 degrees. If we want to convert these Celsius values to Fahrenheit, the formula is $F = 1.8C + 32$. With a NumPy array, the formula applies to every value together.

| Symbol | Meaning | Example value |
| --- | --- | ---: |
| $C$ | Temperature in Celsius | 18
| $1.8$ | Scale factor between the units | 1.8
| $32$ | Fahrenheit offset | 32
| $F$ | Result in Fahrenheit | $1.8(18) + 32 = 64.4$

The code shows the same transformation for all five measurements.

In [ ]:
import numpy as np

celsius_list = [18, 20, 21, 19, 22]
celsius = np.array(celsius_list)
fahrenheit = 1.8 * celsius + 32

print('Celsius:', celsius)
print('Fahrenheit:', fahrenheit)
print('Array type:', type(celsius))

The array expression does not mean that NumPy forgot the five separate readings. It means the same arithmetic is applied to each position. The first output, 64.4, came from the first input, 18; the second output, 68.0, came from 20, and so on. Positions stay aligned.

| Python list | NumPy array |
| --- | --- |
| General-purpose container; values can have different types | Numerical container, usually with one dtype |
| `values + values` joins lists together | `values + values` adds corresponding numbers |
| Arithmetic often needs an explicit loop | Arithmetic is expressed over the full array |
| Good for mixed collections | Good for numerical data and multidimensional shapes |

<div style="background-color:#eaf7ee; border-left:5px solid #39824b; padding:10px 14px; margin:12px 0;">
<strong>Think element by element, not loop by loop.</strong><br>
The operation `1.8 * celsius + 32` keeps the correspondence between input and output positions, while NumPy handles the repeated work.
</div>

## 2. Decide what each dimension represents

Before inspecting shape, name the real-world meaning of each dimension in the example. If the axes were swapped, which summary would change meaning? Array dimensions are part of the data contract, not just a requirement for making code run.

An array is not just a bag of numbers. Its shape tells us how many values sit along each dimension. A single row of readings is one-dimensional; a table of readings from several sensors is two-dimensional.

| Example | Shape | `ndim` | What the dimensions describe |
| --- | --- | ---: | --- |
| `[18, 20, 21]` | `(3,)` | 1 | Three values along one axis |
| `[[18, 20], [19, 21]]` | `(2, 2)` | 2 | Two rows and two columns |
| A stack of 4 tables, each 2 by 3 | `(4, 2, 3)` | 3 | Four groups, two rows per group, three columns per row |

For a two-dimensional array, shape is written `(rows, columns)`. The shape `(2, 3)` means there are 2 rows and 3 columns, so there are $2 \times 3 = 6$ elements.

In [ ]:
sensor_readings = np.array([[18, 20, 21], [19, 22, 23]])

print(sensor_readings)
print('Dimensions:', sensor_readings.ndim)
print('Shape:', sensor_readings.shape)
print('Number of values:', sensor_readings.size)
print('Data type:', sensor_readings.dtype)

Here, `shape` is `(2, 3)`: two sensors and three readings per sensor. `ndim` counts the axes, so this is two-dimensional. `size` counts every stored value, so it is 6. These properties answer different questions and are useful when checking whether arrays can be combined or passed to a function.

<div style="background-color:#fff5df; border-left:5px solid #d28a16; padding:10px 14px; margin:12px 0;">
<strong>Shape errors are often meaning errors.</strong><br>
If a function expects one row per sensor but receives one column per sensor, the numbers may still look valid while the calculation answers the wrong question. Check the shape before doing more work.
</div>

## 3. Create arrays for a particular purpose

Often we are not converting an existing list. We need a range of coordinates, a blank table for results, or equally spaced points for a graph. NumPy has several creation functions; the choice depends on whether we care about a step size or a fixed number of points.

| Function | What we specify | Example output | Useful for |
| --- | --- | --- | --- |
| `np.array(data)` | Existing values | `np.array([2, 4, 6])` | Bring Python data into NumPy |
| `np.arange(start, stop, step)` | Step size | `np.arange(0, 6, 2)` gives `[0, 2, 4]` | Integer-like sequences |
| `np.linspace(start, stop, num)` | Number of values | `np.linspace(0, 1, 5)` gives 5 evenly spaced points | Sampling an interval |
| `np.zeros(shape)` | Shape | `np.zeros((2, 3))` | Initialize numeric output |
| `np.ones(shape)` | Shape | `np.ones(4)` | Start with a constant value of one |
| `np.full(shape, value)` | Shape and fill value | `np.full(3, 7)` | Start with another constant |

`arange` excludes its stop value, like Python's `range`. Floating-point steps can accumulate tiny rounding differences. If we need exactly 5 points including both endpoints, `linspace` expresses that intent directly.

In [ ]:
integer_steps = np.arange(0, 6, 2)
even_points = np.linspace(0, 1, 5)
empty_measurements = np.zeros((2, 3))
one_week = np.ones(7)
baseline = np.full((2, 2), 15.0)

print('arange:', integer_steps)
print('linspace:', even_points)
print('zeros shape:', empty_measurements.shape)
print('ones:', one_week)
print('baseline:\n', baseline)

## 4. A dtype determines how each value is stored

NumPy arrays usually store values using one shared data type, called a dtype. This makes numerical operations predictable and storage efficient. It also means that mixing integers and decimals generally converts the values to a common type.

| Dtype | Example | Stores | Typical use |
| --- | --- | --- | --- |
| Integer, such as `int64` | `7` | Whole numbers | Counts, indexes |
| Floating point, such as `float64` | `7.25` | Approximate real numbers | Measurements, averages |
| Boolean, `bool` | `True` | True/False values | Masks and conditions |
| String, such as `<U...` | `'north'` | Text values | Labels and categories |

A conversion with `.astype(...)` creates values represented in a chosen dtype. Converting a decimal to an integer discards its fractional part; it does not round to the nearest integer. Choose a dtype based on the meaning and precision the calculation needs.

In [ ]:
mixed_numbers = np.array([2, 3.5, 5])
whole_numbers = np.array([2.9, 3.1, 5.8]).astype(np.int64)

print('Mixed dtype:', mixed_numbers.dtype, mixed_numbers)
print('After integer conversion:', whole_numbers)
print('Boolean dtype:', np.array([True, False]).dtype)

<div style="background-color:#fff5df; border-left:5px solid #d28a16; padding:10px 14px; margin:12px 0;">
<strong>Storage type can change the result.</strong><br>
Integer conversion truncates: 2.9 becomes 2. If those decimal parts matter, keep a floating-point dtype instead of converting the array to integers.
</div>

## 5. Select values with indexes and slices

An index is a position along an axis. NumPy uses zero-based indexing: position 0 is the first value. For a 2D array, write one index for each axis as `[row, column]`. A colon means take a range along that axis.

| Expression for `sensor_readings` | Meaning | Shape or result |
| --- | --- | --- |
| `sensor_readings[0, 1]` | First row, second column | One number: `20` |
| `sensor_readings[1]` | Entire second row | Shape `(3,)` |
| `sensor_readings[:, 0]` | First column from every row | Shape `(2,)` |
| `sensor_readings[:, 1:]` | Every row, columns from index 1 onward | Shape `(2, 2)` |

The start of a slice is included and the stop is excluded. Thus `1:3` selects positions 1 and 2. Basic slices commonly produce a view that refers to the original array memory, not an independent copy.

In [ ]:
print('One value:', sensor_readings[0, 1])
print('Second row:', sensor_readings[1])
print('First column:', sensor_readings[:, 0])
print('Columns 2 and 3:\n', sensor_readings[:, 1:])

source_array = np.array([10, 20, 30, 40])
view = source_array[1:3]
view[0] = 99
print('Changing the view also changed the source:', source_array)

independent_copy = source_array[1:3].copy()
independent_copy[0] = -1
print('Changing the copy leaves the source alone:', source_array)

<div style="background-color:#fff5df; border-left:5px solid #d28a16; padding:10px 14px; margin:12px 0;">
<strong>View or copy?</strong><br>
A view shares storage with its source, so changing one can change the other. Use `.copy()` when you need an independent array and do not want later edits to affect the original.
</div>

## 6. Element-wise operations and mathematical products

Array arithmetic usually happens element by element. If a sensor has readings `[2, 4, 6]` and we subtract a baseline `[1, 1, 2]`, each output position subtracts the values at the same position. This is different from multiplying matrices.

| Operation | Meaning | Example |
| --- | --- | --- |
| `a + b` | Add corresponding values | `[2, 4] + [1, 3]` gives `[3, 7]` |
| `a * b` | Multiply corresponding values | `[2, 4] * [1, 3]` gives `[2, 12]` |
| `a @ b` | Matrix product | Combines rows and columns according to linear algebra |
| `np.dot(a, b)` | Dot product or matrix product, depending on dimensions | Commonly used for vector similarity and linear algebra |

The `@` operator is usually clearest for a matrix product. For vectors `[1, 2]` and `[3, 4]`, their dot product is $1\cdot3 + 2\cdot4 = 11$.

In [ ]:
left = np.array([2, 4, 6])
right = np.array([1, 1, 2])
print('Element-wise subtraction:', left - right)
print('Element-wise multiplication:', left * right)

vector_a = np.array([1, 2])
vector_b = np.array([3, 4])
print('Dot product:', vector_a @ vector_b)

matrix_a = np.array([[1, 2], [3, 4]])
matrix_b = np.array([[10, 20], [30, 40]])
print('Element-wise matrix product:\n', matrix_a * matrix_b)
print('Matrix product:\n', matrix_a @ matrix_b)

## 7. Apply an adjustment to the intended axis

Write the two input shapes and predict the broadcast result before executing. Check one output element manually. Compatible shapes alone do not prove that a per-sensor adjustment was applied to sensors rather than time positions.

Suppose each sensor has a different calibration offset. We want to add the offset for sensor A to every reading from A, and the offset for sensor B to every reading from B. Broadcasting lets NumPy align arrays with compatible shapes without manually repeating values.

A 2D array with shape `(3, 2)` has 3 time rows and 2 sensor columns. A baseline array with shape `(2,)` has one offset per sensor. NumPy compares dimensions from the right: sizes match, or one of the sizes must be 1. Here `(3, 2)` and `(2,)` are compatible because the second dimension is 2 and the missing leading dimension acts like 1.

| Time | Sensor A | Sensor B |
| --- | ---: | ---: |
| 1 | 10 | 20 |
| 2 | 12 | 18 |
| 3 | 11 | 21 |

Adding offsets `[1, -2]` means add 1 to every A value and subtract 2 from every B value. The baseline is reused across rows by broadcasting; NumPy does not need us to build a repeated 3-by-2 baseline table.

<div style="background-color:#eaf4fb; border-left:5px solid #2878a5; padding:10px 14px; margin:12px 0;">
<strong>Read broadcasting from the trailing dimensions.</strong><br>
For shapes `(3, 2)` and `(2,)`, compare 2 with 2, then 3 with the missing dimension (treated as 1). The dimensions are compatible, and the result shape is `(3, 2)`. A dimension pair such as 3 and 4 is not compatible.
</div>

In [ ]:
readings = np.array([[10, 20], [12, 18], [11, 21]])
sensor_offsets = np.array([1, -2])
calibrated = readings + sensor_offsets

print('Input shape:', readings.shape)
print('Offset shape:', sensor_offsets.shape)
print('Calibrated readings:\n', calibrated)
print('Result shape:', calibrated.shape)

## 8. Boolean masks select values by condition

Indexing by position answers “Which row or column do I want?”
A boolean mask answers “Which values satisfy this condition?”
 A comparison such as `temperatures > 20` produces a Boolean array with the same shape as `temperatures`. Using that mask as an index selects the matching values.

For example, for temperatures `[18, 21, 24, 19]`, the comparison `> 20` gives `[False, True, True, False]`. The selected values are therefore 21 and 24.

| Tool | What it does | Example use |
| --- | --- | --- |
| `array[condition]` | Returns values where the condition is true | Keep measurements above a threshold |
| `np.where(condition, a, b)` | Chooses between `a` and `b` at each position | Label each value as high or normal |
| `np.clip(array, low, high)` | Limits values to a range | Bound noisy values before a calculation |
| `np.any(condition)` | Checks whether at least one value is true | Is there any reading above a limit? |
| `np.all(condition)` | Checks whether every value is true | Are all readings within a safe range? |

When combining comparisons, use `&` for element-wise AND and `|` for element-wise OR, with parentheses around each comparison. Python's `and` and `or` are for single truth values, not arrays.

In [ ]:
temperatures = np.array([18, 21, 24, 19])
hot_mask = temperatures > 20
print('Mask:', hot_mask)
print('Selected:', temperatures[hot_mask])

labels = np.where(temperatures > 20, 'hot', 'normal')
limited = np.clip(temperatures, 20, 23)
print('Labels:', labels)
print('Clipped:', limited)
print('Any above 22?', np.any(temperatures > 22))
print('All at most 25?', np.all(temperatures <= 25))

comfortable = (temperatures >= 18) & (temperatures <= 22)
print('Comfortable readings:', temperatures[comfortable])

## 9. Choose the direction in which to summarize

Predict how many values remain after each reduction. Calculate one row or column summary by hand and match it to the output. Then explain the difference between a summary per observation and one per feature.

An aggregation reduces several values to a summary. For a table with days in rows and sensors in columns, `sum(axis=0)` reduces the day dimension and returns one total per sensor. `sum(axis=1)` reduces the sensor dimension and returns one total per day. Omitting `axis` summarizes the whole array.

The same axis rules apply to `mean`, `min`, and `max`. Use `keepdims=True` when the reduced dimension should remain length 1 for later broadcasting.


In [ ]:
daily_sensor_data = np.array([[2, 10], [4, 20], [6, 30]])
print('Total per sensor:', daily_sensor_data.sum(axis=0))
print('Total per day:', daily_sensor_data.sum(axis=1))
print('Grand total:', daily_sensor_data.sum())
print('Average per sensor:', daily_sensor_data.mean(axis=0))
print('Largest reading:', daily_sensor_data.max())

## 10. Reshape and transpose without changing the values

Sometimes the same six measurements need to be grouped differently. `reshape` changes the organization of values while preserving their order, as long as the requested shape contains the same number of elements.

A flat array of six values can become shape `(2, 3)` because $2 \times 3 = 6$. It cannot become `(4, 2)`, because that shape needs 8 values. `T` transposes a 2D array by swapping rows and columns.

| Operation | Effect | Original size preserved? |
| --- | --- | --- |
| `reshape(2, 3)` | Reorganizes values into 2 rows and 3 columns | Yes |
| `.T` | Swaps rows and columns | Yes |
| `ravel()` | Returns a flattened 1D view when possible | Yes |
| `flatten()` | Returns a flattened copy | Yes |

The array's shape changes, but its values do not magically become new observations. The order in which they are placed matters.

In [ ]:
measurements = np.arange(1, 7)
grid = measurements.reshape(2, 3)

print('Flat:', measurements)
print('Grid:\n', grid)
print('Transpose:\n', grid.T)
print('Ravel:', grid.ravel())
print('Flatten:', grid.flatten())

## 11. Join arrays along a chosen axis

Suppose one table contains morning readings and another contains evening readings. We can append the evening rows beneath the morning rows, or place them beside the morning columns. The axis determines the direction of the join.

| Function | How arrays are arranged | Shape example |
| --- | --- | --- |
| `np.concatenate((a, b), axis=0)` | Extend along existing rows | `(2, 3)` plus `(1, 3)` becomes `(3, 3)` |
| `np.concatenate((a, b), axis=1)` | Extend along existing columns | `(2, 2)` plus `(2, 1)` becomes `(2, 3)` |
| `np.stack((a, b), axis=0)` | Add a new axis to hold both arrays | Two `(2, 3)` arrays become `(2, 2, 3)` |

For `concatenate`, the dimensions other than the chosen axis must match. `stack` instead requires the input shapes to match and creates a new dimension.

In [ ]:
morning = np.array([[10, 12], [11, 13]])
evening = np.array([[15, 14]])
by_time = np.concatenate((morning, evening), axis=0)
print('Append rows:\n', by_time, 'shape:', by_time.shape)

sensor_a = np.array([[10], [11]])
sensor_b = np.array([[12], [13]])
side_by_side = np.concatenate((sensor_a, sensor_b), axis=1)
print('Append columns:\n', side_by_side)

same_shape_a = np.array([[1, 2, 3]])
same_shape_b = np.array([[4, 5, 6]])
stacked = np.stack((same_shape_a, same_shape_b), axis=0)
print('New stack shape:', stacked.shape)

## 12. Do not let missing measurements become ordinary values

Compare the denominator used by an ordinary mean with one that excludes missing values. An available-value average can be appropriate, but report how much data was absent before treating it as representative.

A missing reading is different from a reading of zero. NumPy commonly represents a missing floating-point value as `np.nan`. NaN means 
 and has special comparison behavior: even `np.nan == np.nan` is False. Use `np.isnan` to identify it.

| Function | Purpose | Example behavior |
| --- | --- | --- |
| `np.isnan(values)` | Find NaN values | Returns a Boolean mask |
| `np.nanmean(values)` | Average while ignoring NaN | Keeps known readings in the calculation |
| `np.isfinite(values)` | Find ordinary finite values | Excludes NaN and positive/negative infinity |
| `np.nan_to_num(values)` | Replace non-finite values | A deliberate conversion, not neutral cleaning |

For `[10, NaN, 14]`, the ordinary mean is NaN because the missing value propagates through arithmetic. The `nanmean` is `(10 + 14) / 2 = 12`. This is appropriate only if ignoring the missing observation matches the question.

<div style="background-color:#fff5df; border-left:5px solid #d28a16; padding:10px 14px; margin:12px 0;">
<strong>Do not replace missing values automatically.</strong><br>
A missing sensor reading can indicate a transmission problem or a meaningful event. Choosing zero, a mean, or a dropped record changes the data and should follow an explicit decision.
</div>

In [ ]:
readings_with_gap = np.array([10.0, np.nan, 14.0, np.inf])
print('NaN positions:', np.isnan(readings_with_gap))
print('Finite positions:', np.isfinite(readings_with_gap))
print('Ordinary mean:', np.mean(readings_with_gap))
print('Mean ignoring NaN:', np.nanmean(readings_with_gap))
print('Replace non-finite values explicitly:', np.nan_to_num(readings_with_gap, nan=-1, posinf=999))

## 13. Generate random values reproducibly

Randomness is useful for simulations, sampling, and randomized tests. NumPy's modern random interface starts with `np.random.default_rng()`, which creates a random number generator. Giving it a seed makes the sequence repeatable, which helps us debug and compare results.

| Generator method | Kind of values | Example question |
| --- | --- | --- |
| `.integers(low, high, size)` | Random integers from `low` up to but not including `high` | Simulate a die roll with values 1 through 6 |
| `.random(size)` | Uniform values from 0 up to 1 | Generate random proportions |
| `.normal(loc, scale, size)` | Values from a normal distribution | Simulate measurement noise around a mean |
| `.choice(values, size, replace=...)` | Sample from provided values | Draw a sample from categories or records |

In `normal(loc=10, scale=2)`, `loc` is the distribution mean (10) and `scale` is its standard deviation (2). A scale of 2 describes typical spread, not a fixed distance from the mean.

In [ ]:
values_with_nan = np.array([10.0, np.nan, 14.0])
readings_with_gap = np.array([10.0, np.nan, 14.0, np.inf])
print('NaN positions:', np.isnan(readings_with_gap))
print('Finite positions:', np.isfinite(readings_with_gap))
print('Ordinary mean:', np.mean(values_with_nan))
print('Mean ignoring NaN:', np.nanmean(values_with_nan))
print('Replace non-finite values explicitly:', np.nan_to_num(readings_with_gap, nan=-1, posinf=999))

## 14. Sort, count, and find positions

Sorting changes the order of values. Sometimes we want the sorted values; sometimes we need the original position of the smallest or largest value. Those are related but different operations.

| Function | Returns | Example use |
| --- | --- | --- |
| `np.sort(values)` | Sorted values | Show readings in ascending order |
| `np.argsort(values)` | Index positions that would sort the values | Keep IDs aligned with sorted scores |
| `np.argmax(values)` | Position of the maximum | Find when the peak reading occurred |
| `np.unique(values, return_counts=True)` | Unique values and their counts | Count category or score frequencies |
| `np.searchsorted(sorted_values, value)` | Insertion position in sorted values | Place a value while preserving order |

For readings `[30, 10, 20]`, `argsort` returns `[1, 2, 0]`: the smallest value is at original position 1, then position 2, then position 0. The indexes let us reorder related arrays in the same way.

In [ ]:
scores = np.array([30, 10, 20, 10, 40])
order = np.argsort(scores)
unique_scores, score_counts = np.unique(scores, return_counts=True)

print('Sorted values:', np.sort(scores))
print('Sort order:', order)
print('Scores in that order:', scores[order])
print('Peak position:', np.argmax(scores))
print('Unique scores:', unique_scores)
print('Counts:', score_counts)
print('Insert 25 at position:', np.searchsorted(np.sort(scores), 25))

## 15. Solve a system of linear equations

Many scientific problems can be written as a system of equations. For example, suppose two unknown quantities $x$ and $y$ satisfy:

$$2x + y = 5, \qquad x + 3y = 6$$

We can write the same system as $A\mathbf{x} = \mathbf{b}$, where:

$$A = \begin{bmatrix}2 & 1 \\ 1 & 3\end{bmatrix}, \qquad \mathbf{x} = \begin{bmatrix}x \\ y\end{bmatrix}, \qquad \mathbf{b} = \begin{bmatrix}5 \\ 6\end{bmatrix}$$

| Symbol | In this example | Meaning |
| --- | --- | --- |
| $A$ | `[[2, 1], [1, 3]]` | Coefficients of the unknowns |
| $\mathbf{x}$ | `[x, y]` | Unknown values we want to find |
| $\mathbf{b}$ | `[5, 6]` | Known right-hand-side values |

`np.linalg.solve(A, b)` computes the solution. For this system, $x=1.8$ and $y=1.4$: the first equation gives $2(1.8)+1.4=5$, and the second gives $1.8+3(1.4)=6$.

<div style="background-color:#eaf4fb; border-left:5px solid #2878a5; padding:10px 14px; margin:12px 0;">
<strong>Solve the system; do not form the inverse just to multiply it.</strong><br>
`np.linalg.solve(A, b)` is the direct tool for a square, nonsingular system. Explicitly computing `inv(A) @ b` is usually less efficient and can be less numerically stable.
</div>

In [ ]:
coefficients = np.array([[2.0, 1.0], [1.0, 3.0]])
known_values = np.array([5.0, 6.0])
solution = np.linalg.solve(coefficients, known_values)

print('x and y:', solution)
print('Check by multiplying A @ solution:', coefficients @ solution)
print('Right-hand side:', known_values)

## 16. Understand memory and performance

NumPy is designed to store numerical values compactly and apply operations across arrays. A value's `itemsize` is the number of bytes used by one element; `nbytes` is the total number of bytes used by the array data. For a 100-element `float64` array, each value uses 8 bytes and the data uses $100 \times 8 = 800$ bytes.

| Property or pattern | What it tells us |
| --- | --- |
| `array.dtype` | How values are represented |
| `array.itemsize` | Bytes per element |
| `array.nbytes` | Bytes used by the element data |
| Vectorized operation | Apply an array operation without a Python-level loop for each item |
| View | A different array shape or slice that can share the same memory |

Vectorized code is often concise and fast for numerical work, but it is not a guarantee that every expression is optimal. Large intermediate arrays also use memory. Measure performance for the actual data size and avoid creating repeated copies without a reason.

In [ ]:
large_array = np.zeros(100, dtype=np.float64)
print('dtype:', large_array.dtype)
print('Bytes per value:', large_array.itemsize)
print('Array data bytes:', large_array.nbytes)

matrix = np.arange(12).reshape(3, 4)
matrix_view = matrix[:, :2]
matrix_copy = matrix[:, :2].copy()
print('View shares memory:', np.shares_memory(matrix, matrix_view))
print('Copy shares memory:', np.shares_memory(matrix, matrix_copy))

## Explain what the evidence supports

Explain one operation through its input shape, output shape, axis meaning and a hand-calculated value. Then identify a shape-compatible operation that would produce the wrong interpretation.

Change one assumption or input at a time. Predict the effect before rerunning, compare it with the actual result, and explain any difference. Keep the original result available so that the comparison is meaningful.

## Second project: recover a sensor summary without fabricating measurements

A 48-sensor array records 1,440 minutes. Dropouts, infinite values and impossible temperatures arrive alongside plausible readings. Compare a tempting zero-fill repair with a validity mask, then compute the same daily report in bounded row chunks. This synthetic scenario includes known corruption so the repair can be checked; it does not claim to reproduce a particular instrument.

Run these cells in order. This extension uses its own `p2_` variables and writes only its named outputs under `outputs/project2/`. Predict each comparison before running it; the worked discussion after the code explains the result and its limits.

### Work through the denominator before scaling up

For readings `[10, missing, 20]`, zero filling produces `(10+0+20)/3=10`; the mean of observed values is `(10+20)/2=15`. Neither formula estimates an unobserved event without an assumption. Use the observed-value mean only when the report is explicitly about available readings, and report coverage next to it. An entirely missing sensor must remain missing.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import time
from IPython.display import display
p2_out=Path('outputs/project2');p2_out.mkdir(parents=True,exist_ok=True)

p2_small=np.array([10.,np.nan,20.])
assert np.nan_to_num(p2_small).mean()==10
assert np.nanmean(p2_small)==15
p2_rng=np.random.default_rng(91)
p2_readings=p2_rng.normal(22,3,(1440,48))
p2_readings[p2_rng.random(p2_readings.shape)<0.08]=np.nan
p2_readings[::71,2]=np.inf;p2_readings[::89,4]=999
p2_readings[:,0]=np.nan
p2_valid=np.isfinite(p2_readings)&(p2_readings>=-40)&(p2_readings<=85)
p2_counts=p2_valid.sum(axis=0)
p2_sums=np.where(p2_valid,p2_readings,0).sum(axis=0)
p2_mean=np.divide(p2_sums,p2_counts,out=np.full(48,np.nan),where=p2_counts>0)
assert np.isnan(p2_mean[0]) and p2_counts[0]==0
p2_wrong=np.nan_to_num(np.where(p2_valid,p2_readings,np.nan)).mean(axis=0)
display(pd.DataFrame({'observed_mean':p2_mean,'zero_fill_mean':p2_wrong,'coverage':p2_counts/1440}).head())

### Work through a bounded-memory alternative

A full validity mask and a full cleaned array both scale with `minutes × sensors`. Accumulate sums and counts in 120-row chunks instead. Do not average chunk means: chunks can have different numbers of valid readings. Add each chunk's numerator and denominator, then divide once. This keeps additional working arrays small, although the source array in this demonstration still occupies memory; a production reader would stream input too.

In [ ]:
p2_start=time.perf_counter()
p2_chunk_sum=np.zeros(48);p2_chunk_count=np.zeros(48,dtype=np.int64)
for p2_start_row in range(0,len(p2_readings),120):
    p2_block=p2_readings[p2_start_row:p2_start_row+120]
    p2_mask=np.isfinite(p2_block)&(p2_block>=-40)&(p2_block<=85)
    p2_chunk_sum+=np.where(p2_mask,p2_block,0).sum(axis=0)
    p2_chunk_count+=p2_mask.sum(axis=0)
p2_chunk_mean=np.divide(p2_chunk_sum,p2_chunk_count,out=np.full(48,np.nan),where=p2_chunk_count>0)
p2_seconds=time.perf_counter()-p2_start
assert np.allclose(p2_mean,p2_chunk_mean,equal_nan=True)
assert np.array_equal(p2_counts,p2_chunk_count)
p2_report=pd.DataFrame({'sensor':np.arange(48),'mean':p2_mean,'coverage':p2_counts/len(p2_readings)})
p2_report.to_csv(p2_out/'numpy_sensor_report.csv',index=False)
print({'source_bytes':p2_readings.nbytes,'chunk_values_bytes':120*48*8,'chunk_seconds':p2_seconds})

### Interpret the repair and try one change

The two correct approaches should agree, including the missing result for sensor 0. The zero-filled estimate should usually be lower because zero is outside the typical temperature range. Rejection bounds detect impossible readings, not every plausible calibration error. Timing here includes Python-loop overhead on a small array and does not establish that chunking is faster.

**Try:** make the first half of sensor 1's readings missing. **Worked reasoning:** its coverage declines, but its observed mean need not decline; denominator and temperature answer different questions. If missingness is concentrated during unusually hot periods, the observed mean is biased for the full-day mean. Report that limitation rather than claiming the mask recovered the missing temperatures.